In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
import kagglehub
from torchvision.models.detection import fasterrcnn_resnet50_fpn 
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch
from torchvision import tv_tensors
from torchvision.models.detection import FasterRCNN
import torchvision
import random
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.datasets import CocoDetection
from torchvision import transforms as T
from torchmetrics.detection import MeanAveragePrecision
from torch.utils.data import Dataset
from torchvision.models import resnet50

path = kagglehub.dataset_download("pkdarabi/bone-fracture-detection-computer-vision-project")

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project


In [2]:
device

'cpu'

# Prepare

### нейросеть с сайта roboflow помогла встроить веса из репозитория RadImageNet в torcvision fasterRCNN. 


In [17]:
import torch
from torchvision.models import resnet50
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import BackboneWithFPN

def extract_state_dict(obj):
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net", "backbone"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj

def normalize_radimagenet_key(k):
    prefixes = [
        "module.",
        "model.",
        "backbone.",
        "encoder.",
        "features.",
    ]

    changed = True
    while changed:
        changed = False
        for p in prefixes:
            if k.startswith(p):
                k = k[len(p):]
                changed = True
    layer_mapping = {
        "0": "conv1",
        "1": "bn1",
        "4": "layer1",
        "5": "layer2",
        "6": "layer3",
        "7": "layer4",
    }
    parts = k.split(".")
    if len(parts) > 0 and parts[0] in layer_mapping:
        parts[0] = layer_mapping[parts[0]]
        k = ".".join(parts)
    return k

def load_radimagenet_into_resnet50(rad_path):
    raw = torch.load(rad_path, map_location="cpu")
    rad_sd = extract_state_dict(raw)

    resnet = resnet50(weights=None)
    target_sd = resnet.state_dict()

    converted = {}
    skipped = []
    for k, v in rad_sd.items():
        nk = normalize_radimagenet_key(k)
        if nk.startswith("fc."):
            skipped.append((k, nk, "fc skipped"))
            continue
        if nk == "conv1.weight":
            if v.ndim == 4 and v.shape[1] == 1 and target_sd[nk].shape[1] == 3:
                v = v.repeat(1, 3, 1, 1) / 3.0
        if nk in target_sd and target_sd[nk].shape == v.shape:
            converted[nk] = v
        else:
            reason = "not in target" if nk not in target_sd else f"shape mismatch {tuple(v.shape)} != {tuple(target_sd[nk].shape)}"
            skipped.append((k, nk, reason))

    missing, unexpected = resnet.load_state_dict(converted, strict=False)

    print("Loaded keys:", len(converted))
    print("Skipped keys:", len(skipped))
    print("Missing keys count:", len(missing))
    print("Unexpected keys:", unexpected)

    print("\nFirst 20 missing:")
    for x in missing[:20]:
        print(" ", x)

    print("\nFirst 20 skipped:")
    for old, new, reason in skipped[:20]:
        print(f"  {old} -> {new}: {reason}")

    # coverage по основным слоям
    print("\nCoverage:")
    for prefix in ["conv1", "bn1", "layer1", "layer2", "layer3", "layer4"]:
        loaded = sum(k.startswith(prefix) for k in converted)
        total = sum(k.startswith(prefix) for k in target_sd)
        print(f"{prefix}: {loaded}/{total}")

    return resnet


In [18]:
with torch.no_grad():
    w = resnet.conv1.weight.data
    resnet.conv1.weight.data = w.mean(dim=1, keepdim=True).repeat(1, 3, 1, 1) / 3.0


In [19]:
x = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    y = resnet.conv1(x)
print(y.mean().item(), y.std().item())


-0.00020359823247417808 0.4109908640384674


In [4]:
rad_path = "/kaggle/input/models/fiktus/resnte5rad/pytorch/default/1/ResNet50.pt"

resnet = load_radimagenet_into_resnet50(rad_path)

return_layers = {
    "layer1": "0",
    "layer2": "1",
    "layer3": "2",
    "layer4": "3",
}

backbone = BackboneWithFPN(
    resnet,
    return_layers=return_layers,
    in_channels_list=[256, 512, 1024, 2048],
    out_channels=256,
)

model = FasterRCNN(backbone, num_classes=8)


Loaded keys: 318
Skipped keys: 0
Missing keys count: 2
Unexpected keys: []

First 20 missing:
  fc.weight
  fc.bias

First 20 skipped:

Coverage:
conv1: 1/1
bn1: 5/5
layer1: 60/60
layer2: 78/78
layer3: 114/114
layer4: 60/60


In [6]:
#rad_weights = torch.load('/kaggle/input/models/fiktus/resnte5rad/pytorch/default/1/ResNet50.pt', 
#                        map_location='cpu')
#rad_weights = convert_radimagenet_keys(rad_weights)

#model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

#model.roi_heads.box_predictor = FastRCNNPredictor(in_channels=1024, num_classes=8)#7+1 pod fon

#model.score_thresh = 0.0 
#model.roi_heads.box_predictor.score_thresh = 0.0

In [7]:
#model.roi_heads.box_predictor

# Data

In [8]:
class_names = ['elbow positive', 'fingers positive', 'forearm fracture', 'humerus fracture', 'humerus', 'shoulder fracture', 'wrist positive']

def prline(line, img_w, img_h):
    parts = list(map(float, line.strip().split()))
    cls = int(parts[0])
    coords = parts[1:]
    if len(coords) == 4:        
        cx, cy, bw, bh = coords
        x1 = (cx - bw/2) * img_w
        y1 = (cy - bh/2) * img_h
        x2 = (cx + bw/2) * img_w
        y2 = (cy + bh/2) * img_h
    else:                   
        xs = coords[0::2]         
        ys = coords[1::2]         
        x1 = min(xs) * img_w
        y1 = min(ys) * img_h
        x2 = max(xs) * img_w
        y2 = max(ys) * img_h
    return cls, [x1, y1, x2, y2]
    
class custDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        
        self.data = []
        for lf in sorted(os.listdir(labels_dir)):
            img_path = os.path.join(images_dir, lf[:-4] + '.jpg')
            if os.path.exists(img_path):
                self.data.append((img_path, os.path.join(labels_dir, lf)))
        
        self.image_files = [d[0] for d in self.data]
        self.label_files = [d[1] for d in self.data]

                
            
    def __len__(self):
        return len(self.label_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        label_path = self.label_files[idx]
        boxes = []
        labels = []
        with open(label_path, 'r') as f:
            for line in f:
                if not line.strip():
                    continue
                cls, bbox = prline(line, w, h)
                boxes.append(bbox)
                labels.append(cls + 1)  
        if len(boxes) == 0:
            boxes = torch.empty((0, 4))
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
            labels = torch.tensor(labels, dtype=torch.int64)
        boxes = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(h, w))
        #labels = torch.tensor(labels, dtype=torch.int64)
        
        target = {
            'boxes': boxes,
            'labels': labels,
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target
        
def collate_fn(batch):
    return tuple(zip(*batch))

In [9]:
from torchvision.transforms import v2
def get_train_transform():
    return v2.Compose([
        v2.ToImage(),
        ####
        v2.RandomHorizontalFlip(p=0.3),          
        v2.RandomRotation(degrees=10),
        v2.RandomAffine(degrees = 0, scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),
        #############
        v2.SanitizeBoundingBoxes(min_size=1),  
        v2.ToDtype(torch.float32, scale=True),
        
    ])

def get_val_transform():
    return v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
    ])

# Train methods

In [10]:
from tqdm import tqdm
def train_one_epoch(model, optim, train_dataloader):
    model.train()
    ttlloss = 0
    n = 0
    for images, targets in train_dataloader:
        optim.zero_grad()
        images = [i.to(device) for i in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss = model(images, targets)
        loss = sum(loss.values())
        
        ttlloss += loss.item()
        n += 1
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optim.step()
        
    return ttlloss/n

        
        

In [11]:
def test_one_epoch(model, test_dataloader, metr):
    model.eval()
    metr.reset()
    with torch.no_grad():
        for images, targets in test_dataloader:
            images = [i.to(device) for i in images]
            preds = model(images)
            predscpu = []
            for p in preds:
                mask = p['scores'] >= 0.00
                predscpu.append({
                    'boxes': p['boxes'][mask].cpu(),
                    'scores': p['scores'][mask].cpu(),
                    'labels': p['labels'][mask].cpu()
                })
            targets = [{
                'boxes': t['boxes'], 
                'labels': t['labels']} for t in targets] 
            metr.update(predscpu, targets)
    return metr.compute()

In [12]:
import copy
def train_model(model, train_dataloader, test_dataloader, optim, scheduler, metr, epochs):
    weights = copy.deepcopy(model.state_dict())
    global_map = 0
    loss_stat = []
    map_stat = []
    for i, ep in tqdm(enumerate(range(epochs))):

        curr_loss = train_one_epoch(model, optim, train_dataloader)
        loss_stat.append(curr_loss)
        
        curr_test = test_one_epoch(model, test_dataloader, metr)
        curr_map = curr_test['map_50'].item()
        map_stat.append(curr_map)
        
        lrbef = optim.param_groups[0]['lr']
        scheduler.step(curr_map)
        lrcurr = optim.param_groups[0]['lr']
        
        if lrbef != lrcurr:
            print(f'current lr: {lrcurr}, last lr: {lrbef}')
        if curr_map > global_map:
            global_map = curr_map
            weights = copy.deepcopy(model.state_dict())
            
        print(f'current loss: {curr_loss:.4f}, current_map: {curr_map:.4f}')
        per_class = curr_test['map_per_class']
        print(f'Per-class AP: {per_class}')

        if (i+1)%5 == 0:
            torch.save(weights, 'checkp.pth')
    model.load_state_dict(weights)
    
    return loss_stat, map_stat

# Train

In [13]:
train_ds = custDataset('/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/train/images', 
                       '/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/train/labels',
                       transforms=get_train_transform())
valid_ds = custDataset('/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/valid/images', 
                       '/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/valid/labels',
                       transforms=get_val_transform())
test_ds = custDataset('/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/test/images', 
                       '/kaggle/input/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/BoneFractureYolo8/test/labels',
                       transforms=get_val_transform())

In [14]:
from torch.utils.data import WeightedRandomSampler
weights = []
for idx in range(len(train_ds)):
    _, target = train_ds[idx]
    labels = target['labels'].tolist()
    if 4 in labels:
        weights.append(40)
    else:
        weights.append(1)

sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size = 2, sampler=sampler, num_workers = 4, collate_fn = collate_fn)
valid_loader = torch.utils.data.DataLoader(valid_ds, batch_size = 2, num_workers = 4, collate_fn = collate_fn)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size = 2, num_workers = 4, collate_fn = collate_fn)

In [17]:
metric = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
backbone_params = []   # layer2 layer3 layer4
head_params = []       # FPN RPN ROI
model.to(device)
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith('backbone.body.layer'):
        backbone_params.append(p)
    else:
        head_params.append(p)

params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params, 
    lr=0.005,      
    momentum=0.9,   
    weight_decay=0.0005 #
)
lr_scheduler_full = torch.optim.lr_scheduler.ReduceLROnPlateau( optimizer, mode='max', factor=0.2, patience=3)



In [18]:
train_stat_st2 = train_model(model, train_loader, valid_loader, optimizer, lr_scheduler_full, metric, 20)

1it [23:36, 1416.86s/it]

current loss: 0.1604, current_map: 0.0057
Per-class AP: tensor([ 3.3099e-04,  2.0877e-02,  1.8407e-03, -1.0000e+00,  7.1564e-03,
         3.5017e-03,  3.6187e-04])


2it [47:27, 1425.17s/it]

current loss: 0.1568, current_map: 0.0192
Per-class AP: tensor([ 9.4299e-04,  5.2007e-03,  3.2786e-02, -1.0000e+00,  4.4647e-02,
         2.9346e-02,  2.3389e-03])


3it [1:11:23, 1429.78s/it]

...

18it [7:07:14, 1419.28s/it]

current loss: 0.1211, current_map: 0.1018
Per-class AP: tensor([ 0.0402,  0.1265,  0.2114, -1.0000,  0.1335,  0.0896,  0.0097])


19it [7:31:00, 1421.37s/it]

current loss: 0.1180, current_map: 0.1092
Per-class AP: tensor([ 0.0274,  0.1333,  0.2205, -1.0000,  0.1636,  0.1037,  0.0069])
current loss: 0.1194, current_map: 0.1029
Per-class AP: tensor([ 0.0271,  0.1271,  0.2119, -1.0000,  0.1388,  0.1050,  0.0077])


20it [7:54:52, 1424.60s/it]


In [19]:
torch.save(model.state_dict(), 'best_fracture_model.pth')